<a href="https://colab.research.google.com/github/purnabandekar/TNS/blob/main/Evaluation_Metrics_Day12_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluation Metrics — Day 12 Practical

### Python Course · Colab Notebook

---

**How to use this notebook**

1. Run every cell in order, top to bottom. Click the cell and press **Shift + Enter**.
2. **Every line of code has a comment** explaining what it does.
3. Every section ends with a **YOUR TURN** box. Try it before moving on.
4. **Section 10 is a set of exercises with worked answers.**
5. Nothing to install and nothing to upload.

**To save your own copy:** File → Save a copy in Drive.

---

### What we are covering

| Section | Topic |
|---|---|
| 1 | The 100 emails |
| 2 | Counting the four boxes yourself |
| 3 | The confusion matrix |
| 4 | Accuracy |
| 5 | When accuracy lies |
| 6 | Precision |
| 7 | Recall |
| 8 | F1 |
| 9 | Strict filter vs loose filter |
| 10 | **Exercises**, with answers |

---

The whole notebook uses the same spam filter from the slides: **100 emails, 40 of them
really spam.** Nothing changes, so you can always check your numbers against the deck.

---
## Section 1 — The 100 emails

We need two lists:

- **`y_true`** — what each email really was. This is the truth.
- **`y_pred`** — what the filter guessed.

We write **1 for spam** and **0 for not spam**. Spam is the thing we are hunting for, so
spam is the **positive** class.

In [ ]:
y_true = [1] * 40 + [0] * 60          # the TRUTH: 40 real spams, then 60 normal emails
                                       # [1] * 40 means "a list with 1 in it, forty times"

y_pred = [1] * 30 + [0] * 10 + [1] * 5 + [0] * 55   # what the FILTER guessed, lined up with y_true
                                       # first 30 real spams: correctly called spam
                                       # next 10 real spams: wrongly called not-spam
                                       # next 5 normal emails: wrongly called spam
                                       # last 55 normal emails: correctly called not-spam

print("emails in total :", len(y_true))       # len() counts the items in a list
print("real spams      :", sum(y_true))       # sum() of a 0/1 list counts the 1s
print("filter said spam:", sum(y_pred))       # how many it moved to the junk folder

The two lists are **lined up**. Position 0 in `y_true` and position 0 in `y_pred` are the
same email. That is what lets us compare them.

> **YOUR TURN**
>
> How many emails did the filter leave in the inbox? Work it out from the two numbers printed
> above, then check with `len(y_pred) - sum(y_pred)`.

In [ ]:
# write your answer here

---
## Section 2 — Counting the four boxes yourself

Before we let Python do it, let us count the four boxes by hand. Every email falls into
exactly one of them.

| | filter said NOT spam | filter said SPAM |
|---|---|---|
| **really not spam** | true negative | false positive |
| **really SPAM** | false negative | true positive |

In [ ]:
tp = 0                                 # true positives  - said spam, really spam
tn = 0                                 # true negatives  - said not spam, really not spam
fp = 0                                 # false positives - said spam, really NOT spam
fn = 0                                 # false negatives - said not spam, really SPAM

for i in range(len(y_true)):           # go through the emails one at a time, by position
    real = y_true[i]                   # what this email really was
    guess = y_pred[i]                  # what the filter said about it

    if real == 1 and guess == 1:       # really spam, and it said spam
        tp = tp + 1                    # so add one to true positives
    if real == 0 and guess == 0:       # really not spam, and it said not spam
        tn = tn + 1                    # so add one to true negatives
    if real == 0 and guess == 1:       # really not spam, but it said spam
        fp = fp + 1                    # so add one to false positives
    if real == 1 and guess == 0:       # really spam, but it said not spam
        fn = fn + 1                    # so add one to false negatives

print("true positives  (caught spam)        :", tp)      # should be 30
print("true negatives  (left normal alone)  :", tn)      # should be 55
print("false positives (binned a real email):", fp)      # should be 5
print("false negatives (missed a spam)      :", fn)      # should be 10
print()                                                  # a blank line, for readability
print("all four added up:", tp + tn + fp + fn)           # must equal 100

**30, 55, 5, 10 — and they add up to 100.** Every email is in exactly one box, so they
always must.

Those four numbers are all you need. Every metric today is just arithmetic on them.

> **YOUR TURN**
>
> Which mistake did this filter make more often — binning a real email, or letting a spam
> through? Look at `fp` and `fn`.

In [ ]:
# write your answer here

---
## Section 3 — The confusion matrix

That 2×2 table has a name: the **confusion matrix**. scikit-learn builds it for you.

In [ ]:
from sklearn.metrics import confusion_matrix    # the function that builds the table

cm = confusion_matrix(y_true, y_pred)           # TRUE answers first, PREDICTIONS second
                                                # get this order wrong and everything flips

print(cm)                                       # a 2x2 array

The layout is always the same, and it is worth memorising:

```
[[ true negative    false positive ]
 [ false negative   true positive  ]]
```

So the top-left and bottom-right are the ones it got right.

In [ ]:
print("true negatives :", cm[0][0], "   we counted", tn)   # row 0, column 0
print("false positives:", cm[0][1], "    we counted", fp)  # row 0, column 1
print("false negatives:", cm[1][0], "   we counted", fn)   # row 1, column 0
print("true positives :", cm[1][1], "   we counted", tp)   # row 1, column 1

Identical to our hand count. Now draw it, because a picture is much easier to read out to
somebody else.

In [ ]:
import matplotlib.pyplot as plt         # for drawing - Day 8
import seaborn as sns                   # nicer looking charts - Day 8

plt.figure(figsize=(5.5, 4))                                     # a small picture
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,   # annot writes the numbers in,
            xticklabels=["said NOT spam", "said SPAM"],          # fmt="d" shows them as whole numbers
            yticklabels=["really not spam", "really SPAM"])      # the two lists are the axis labels
plt.title("Confusion matrix - 100 emails")                       # a title
plt.tight_layout()                                               # stop the labels being cut off
plt.show()                                                       # draw it

> **YOUR TURN**
>
> Point at the square that means "a real email ended up in the junk folder". What number is
> in it?

In [ ]:
# write your answer here

---
## Section 4 — Accuracy

**How often was the filter right, overall?**

$$\text{accuracy} = \frac{\text{the ones it got right}}{\text{all of them}}$$

In [ ]:
right = tp + tn                        # the two boxes where it was correct
total = tp + tn + fp + fn              # all four boxes, so all 100 emails

accuracy_by_hand = right / total       # divide one by the other

print("got right:", right, "out of", total)               # 85 out of 100
print("accuracy :", accuracy_by_hand)                     # 0.85
print("as a %   :", round(accuracy_by_hand * 100, 1), "%")   # 85.0 %

In [ ]:
from sklearn.metrics import accuracy_score   # the library version

accuracy_sklearn = accuracy_score(y_true, y_pred)   # true answers first, predictions second

print("our answer    :", accuracy_by_hand)          # what we worked out
print("sklearn answer:", accuracy_sklearn)          # what the library says
print("same?         :", accuracy_by_hand == accuracy_sklearn)   # compare them

Identical. There is nothing hidden inside `accuracy_score` — it counts the same two boxes
you just counted.

**In words:** *"Out of every 100 emails, this filter gets 85 right."*

> **YOUR TURN**
>
> If the filter had got 5 more emails right, what would the accuracy be?

In [ ]:
# write your answer here

---
## Section 5 — When accuracy lies

Now the important part. Accuracy is easy to understand, and easy to be fooled by.

A disease that only 1 person in 100 has. We test 1000 people, so about 10 of them have it.
And we build the laziest model possible: **it always says "healthy"**.

In [ ]:
disease_true = [1] * 10 + [0] * 990    # the truth: 10 sick people, 990 healthy ones
lazy_pred = [0] * 1000                 # our lazy model: it says "healthy" for everybody

print("people tested:", len(disease_true))    # 1000
print("actually sick:", sum(disease_true))    # 10
print("model found  :", sum(lazy_pred))       # 0 - it never says anybody is sick

In [ ]:
lazy_accuracy = accuracy_score(disease_true, lazy_pred)   # score the lazy model

print("accuracy:", round(lazy_accuracy * 100, 1), "%")    # 99.0 %

**99% accurate.** A model that has never once found a sick person.

It is right about all 990 healthy people, and wrong about only 10. So accuracy looks
brilliant — and the model is completely useless for the one job it had.

This is the single most important idea in the whole session.

In [ ]:
from sklearn.metrics import recall_score       # the score that catches this

lazy_recall = recall_score(disease_true, lazy_pred)   # score the lazy model on recall

print("accuracy:", round(lazy_accuracy * 100, 1), "%")    # 99.0 % - looks great
print("recall  :", round(lazy_recall * 100, 1), "%")      # 0.0 % - the truth

Recall asks a different question: *of the people who really were sick, how many did we
find?* The answer is none.

**The rule:** when one answer is much rarer than the other, never quote accuracy on its own.

> **YOUR TURN**
>
> Suppose the disease affected 1 person in 1000 instead of 1 in 100. What accuracy would the
> same lazy model get?

In [ ]:
# write your answer here

---
## Section 6 — Precision

Back to our spam filter.

**When the filter says "spam", how often is it right?**

Look only at the emails it moved to the junk folder — that is `tp + fp`.

$$\text{precision} = \frac{\text{spam it caught}}{\text{everything it CALLED spam}}$$

In [ ]:
called_spam = tp + fp                  # everything it moved to the junk folder: 30 + 5 = 35

precision_by_hand = tp / called_spam   # of those 35, how many really were spam?

print("it called", called_spam, "emails spam")                 # 35
print("of those,", tp, "really were spam")                     # 30
print("precision:", round(precision_by_hand, 4))               # 0.8571
print("as a %   :", round(precision_by_hand * 100, 1), "%")    # 85.7 %

In [ ]:
from sklearn.metrics import precision_score    # the library version

print("our answer    :", round(precision_by_hand, 6))                      # ours
print("sklearn answer:", round(precision_score(y_true, y_pred), 6))        # the library's

**In words:** *"When this filter says spam, it is right about 86% of the time."*

**Low precision means** your junk folder is full of real emails. That is the mistake
precision is watching for.

> **YOUR TURN**
>
> If the filter had made 10 false positives instead of 5 (and everything else stayed the
> same), what would precision become? Work it out with `tp / (tp + 10)`.

In [ ]:
# write your answer here

---
## Section 7 — Recall

**Of all the spam that actually arrived, how much did the filter catch?**

Look only at the emails that really were spam — that is `tp + fn`.

$$\text{recall} = \frac{\text{spam it caught}}{\text{all the real spam}}$$

In [ ]:
real_spam = tp + fn                    # everything that really was spam: 30 + 10 = 40

recall_by_hand = tp / real_spam        # of those 40, how many did it catch?

print("there were", real_spam, "real spams")                # 40
print("it caught", tp, "of them")                           # 30
print("recall :", round(recall_by_hand, 4))                 # 0.75
print("as a % :", round(recall_by_hand * 100, 1), "%")      # 75.0 %

In [ ]:
print("our answer    :", round(recall_by_hand, 6))                   # ours
print("sklearn answer:", round(recall_score(y_true, y_pred), 6))     # the library's

**In words:** *"Of all the spam that arrived, this filter caught 75% of it."*

**Low recall means** spam is still landing in your inbox. That is the mistake recall is
watching for.

### The one thing students mix up

Both use `tp` on the top. What changes is the **bottom**:

| | bottom of the fraction | the question it answers |
|---|---|---|
| **Precision** | everything it CALLED spam | can I trust it when it says yes? |
| **Recall** | all the REAL spam | did it find everything? |

> **YOUR TURN**
>
> Our filter has precision 86% and recall 75%. In one sentence, what does that tell you about
> the kind of mistakes it makes?

In [ ]:
# write your answer here

---
## Section 8 — F1

Sometimes you need **one** number — to put in a report, or to compare two models.

F1 combines precision and recall. It is deliberately harsh: it is only high when **both**
are high.

$$F1 = \frac{2 \times \text{precision} \times \text{recall}}{\text{precision} + \text{recall}}$$

In [ ]:
top = 2 * precision_by_hand * recall_by_hand        # the top of the fraction
bottom = precision_by_hand + recall_by_hand         # the bottom of the fraction

f1_by_hand = top / bottom                           # divide one by the other

print("precision:", round(precision_by_hand, 4))    # 0.8571
print("recall   :", round(recall_by_hand, 4))       # 0.75
print("F1       :", round(f1_by_hand, 4))           # 0.8

In [ ]:
from sklearn.metrics import f1_score                # the library version

print("our answer    :", round(f1_by_hand, 6))                   # ours
print("sklearn answer:", round(f1_score(y_true, y_pred), 6))     # the library's

### Why not just take the average?

Here is the reason, in one cell. Imagine a filter that flags **exactly one** email as spam,
and is right about it. Perfect precision — and it missed the other 39 spams. Useless.

In [ ]:
fake_precision = 1 / 1                               # it flagged 1 email and was right: perfect
fake_recall = 1 / 40                                 # but there were 40 real spams, so it found 1

plain_average = (fake_precision + fake_recall) / 2   # the ordinary average of the two

fake_f1 = 2 * fake_precision * fake_recall / (fake_precision + fake_recall)   # the F1 formula

print("precision    :", round(fake_precision, 3))    # 1.0
print("recall       :", round(fake_recall, 3))       # 0.025
print("plain average:", round(plain_average, 3))     # 0.512 - sounds half decent!
print("F1           :", round(fake_f1, 3))           # 0.049 - the honest answer

A plain average says 51%, which sounds like a half-decent model. F1 says **5%**, which is the
truth. That is exactly why F1 exists — it will not let one perfect score cover for a terrible
one.

### All four at once

`classification_report` prints everything in one table.

In [ ]:
from sklearn.metrics import classification_report    # everything in one printed table

print(classification_report(y_true, y_pred,          # true answers first, predictions second
                            target_names=["not spam", "spam"]))   # readable row names

Read it **by row**, not by column. The `spam` row is the one we have been calculating all
session: precision 0.86, recall 0.75, f1 0.80.

`support` just means how many emails were really in that class — 40 spam, 60 not spam.

**Ignore the `macro avg` and `weighted avg` rows for now.** They are other ways of averaging
the two classes together, and they are not part of today.

> **YOUR TURN**
>
> Find the three numbers we worked out by hand (0.86, 0.75, 0.80) in the report. Which row are
> they on, and why that row?

In [ ]:
# write your answer here

---
## Section 9 — Strict filter vs loose filter

Now let us see the trade-off from the slides, in real numbers.

Same 100 emails. Three different filters:

- our **normal** filter
- a **strict** one that only bins an email when it is very sure
- a **loose** one that bins anything slightly suspicious

In [ ]:
strict_pred = [1] * 19 + [0] * 21 + [1] * 1 + [0] * 59    # cautious: bins only 20 emails
loose_pred = [1] * 38 + [0] * 2 + [1] * 22 + [0] * 38     # eager: bins 60 emails

print("normal filter bins:", sum(y_pred), "emails")        # 35
print("strict filter bins:", sum(strict_pred), "emails")   # 20
print("loose  filter bins:", sum(loose_pred), "emails")    # 60

In [ ]:
print("NORMAL filter")                                                    # our original filter
print("  accuracy :", round(accuracy_score(y_true, y_pred), 3))           # 0.85
print("  precision:", round(precision_score(y_true, y_pred), 3))          # 0.857
print("  recall   :", round(recall_score(y_true, y_pred), 3))             # 0.75
print()                                                                   # blank line

print("STRICT filter")                                                    # only bins when very sure
print("  accuracy :", round(accuracy_score(y_true, strict_pred), 3))      # 0.78
print("  precision:", round(precision_score(y_true, strict_pred), 3))     # 0.95
print("  recall   :", round(recall_score(y_true, strict_pred), 3))        # 0.475
print()                                                                   # blank line

print("LOOSE filter")                                                     # bins anything suspicious
print("  accuracy :", round(accuracy_score(y_true, loose_pred), 3))       # 0.76
print("  precision:", round(precision_score(y_true, loose_pred), 3))      # 0.633
print("  recall   :", round(recall_score(y_true, loose_pred), 3))         # 0.95

Read the **precision** and **recall** columns together, going down:

- The **strict** filter has the best precision (0.95) and the worst recall (0.475). Almost
  everything it bins really is spam, but it lets more than half the spam through.
- The **loose** filter is the exact opposite: recall 0.95, precision 0.633. It catches nearly
  all the spam, but 22 real emails ended up in the junk folder.
- The **normal** filter sits in between, and has the best F1.

Notice how little **accuracy** moves: 0.85, 0.78, 0.76. Meanwhile precision swings from 0.63
to 0.95 and recall from 0.475 to 0.95. These three filters behave completely differently, and
accuracy alone would barely tell you.

> **YOUR TURN**
>
> Which of the three filters would you actually install on your email? There is no single
> right answer — say which, and why.

In [ ]:
# write your answer here

---
## Section 10 — Exercises

Try these before looking at the answers. They use variables from earlier sections, so run
the whole notebook first.

**1.** A model was tested on 200 items. It got `tp = 60`, `tn = 100`, `fp = 20`, `fn = 20`.
Work out its accuracy, precision, recall and F1 using only those four numbers.

**2.** Using `y_true` and `y_pred`, print the confusion matrix and then print a sentence
saying how many real emails ended up in the junk folder.

**3.** Build a filter that flags **every** email as spam (a list of 100 ones). What are its
precision and recall? Explain why.

**4.** Build a filter that flags **no** email as spam. What is its accuracy, and what is its
recall? Which number tells the truth about it?

**5.** Out of the three filters in Section 9, which one would a hospital want if the "positive"
class were a disease? Print your reasoning as a comment.

In [ ]:
# exercise 1

In [ ]:
# exercise 2

In [ ]:
# exercise 3

In [ ]:
# exercise 4

In [ ]:
# exercise 5

---
### Answers

Run these only after you have tried.

In [ ]:
# ANSWER 1
e_tp = 60                              # the four numbers we were given
e_tn = 100                             # true negatives
e_fp = 20                              # false positives
e_fn = 20                              # false negatives

e_acc = (e_tp + e_tn) / (e_tp + e_tn + e_fp + e_fn)   # right ones divided by everything
e_prec = e_tp / (e_tp + e_fp)                         # of the ones it called positive
e_rec = e_tp / (e_tp + e_fn)                          # of the ones that really were positive
e_f1 = 2 * e_prec * e_rec / (e_prec + e_rec)          # the F1 formula

print("accuracy :", round(e_acc, 3))    # 0.8
print("precision:", round(e_prec, 3))   # 0.75
print("recall   :", round(e_rec, 3))    # 0.75
print("F1       :", round(e_f1, 3))     # 0.75

In [ ]:
# ANSWER 2
cm2 = confusion_matrix(y_true, y_pred)              # build the table again

print(cm2)                                          # show it
print()                                             # blank line
print(cm2[0][1], "real emails ended up in the junk folder.")   # row 0, column 1 is the false positives

In [ ]:
# ANSWER 3
flag_all = [1] * 100                                # say "spam" about every single email

print("precision:", round(precision_score(y_true, flag_all), 3))   # 0.4
print("recall   :", round(recall_score(y_true, flag_all), 3))      # 1.0
print()                                                            # blank line
print("Recall is 1.0 - it caught every real spam, because it caught everything.")   # why recall is perfect
print("Precision is 0.4 - only 40 of the 100 it flagged were really spam.")        # why precision is poor

In [ ]:
# ANSWER 4
flag_none = [0] * 100                               # say "not spam" about every single email

print("accuracy:", round(accuracy_score(y_true, flag_none), 3))              # 0.6
print("recall  :", round(recall_score(y_true, flag_none), 3))                 # 0.0
print()                                                                       # blank line
print("Accuracy of 0.6 makes it sound half decent.")            # the misleading number
print("Recall of 0.0 is the truth: it never caught one spam.")  # the honest number

In [ ]:
# ANSWER 5
print("A hospital would choose the LOOSE filter.")                      # the answer
print()                                                                 # blank line
print("Its recall is 0.95, so it finds almost every real case.")        # the reason
print("Its precision is only 0.633, so there are many false alarms -")   # the cost of choosing it
print("but a false alarm just means one more test, while a missed")      # continued
print("case could be fatal. The two mistakes do not cost the same.")     # the real reason

---
## What to remember

| | |
|---|---|
| The four boxes | TP, TN, FP, FN — every prediction is in exactly one |
| Accuracy | right ones ÷ everything |
| Precision | TP ÷ everything it CALLED positive |
| Recall | TP ÷ everything that REALLY was positive |
| F1 | one number, high only when both are high |
| The trap | when one class is rare, accuracy alone means nothing |
| The trade-off | stricter = better precision, worse recall |
| The order | always `metric(y_true, y_pred)` — truth first |

### The mistakes that cost marks

1. Passing the predictions first and the truth second.
2. Confusing precision and recall — remember, only the **bottom** of the fraction changes.
3. Quoting accuracy on an imbalanced dataset and stopping there.
4. Reading `classification_report` down the columns instead of across the rows.

---

**In one sentence:** accuracy says how often you were right, precision says whether to trust
a yes, recall says whether you found them all, and F1 refuses to let one of those cover for
the other.